In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import JsonOutputParser

In [6]:
# 1. Initialize LLM
llm = ChatOllama(
    model="mistral",
    temperature=0.7
)

In [7]:
# 1. Initialize the Parser
parser = JsonOutputParser()

In [8]:
# 2. Create the Prompt (CRITICAL: Inject format instructions)
template = """
Extract the user details from the following text.
Text: {text}

{format_instructions}
"""

In [9]:
prompt = PromptTemplate(
    template=template,
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()} 
)

In [10]:
# 3. Build Chain
chain = prompt | llm | parser

In [11]:
# 4. Execute
text_input = "Gagan is a 24 year old software engineer living in Bengaluru."
result = chain.invoke({"text": text_input})

In [12]:
print(type(result)) # Output: <class 'dict'>
print(result)

<class 'dict'>
{'Name': 'Gagan', 'Age': '24', 'Occupation': 'software engineer', 'Location': 'Bengaluru'}


In [13]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [14]:
# 1. Define the Schema using Pydantic
class DSAConcept(BaseModel):
    concept_name: str = Field(description="Name of the data structure or algorithm")
    time_complexity: str = Field(description="Big O time complexity")
    is_linear: bool = Field(description="True if it is a linear data structure")
    is_tree: bool = Field(description="True if it is a tree data structure")

In [15]:
# 2. Initialize Parser with Schema
pydantic_parser = PydanticOutputParser(pydantic_object=DSAConcept)

In [19]:
prompt = PromptTemplate(
    template="""
You are a DSA expert.

Explain the given concept and return ONLY a JSON object.

Concept: {query}

Rules:
- Return only JSON.
- Do not return JSON schema.
- Do not use "properties", "title", or "type" keys.
- Ensure boolean values are true or false (without quotes).

{format_instructions}
""",
    input_variables=["query"],
    partial_variables={
        "format_instructions": pydantic_parser.get_format_instructions()
    }
)

In [20]:
# 4. Build and Invoke Chain
chain = prompt | llm | pydantic_parser

In [21]:
result = chain.invoke({"query": "Explain Queues"})

In [22]:

print(type(result)) # Output: <class '__main__.DSAConcept'>
print(f"Name: {result.concept_name}, Linear: {result.is_linear}")

<class '__main__.DSAConcept'>
Name: Queues, Linear: False
